<a href="https://colab.research.google.com/github/Crnkcbs/money_fraud/blob/main/Money_Fraud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BIGQUERY'DEN DOSYA YÜKLEME

##Kütüphaneler Yüklendi

In [3]:
from google.colab import auth
from google.colab import drive
from google.cloud import bigquery
import pandas as pd

# 1. Google Drive'ınızı Colab'e sanal bir klasör olarak bağlayın
drive.mount('/content/drive')

# 2. BigQuery'e erişebilmek için Google hesabınızla kimlik doğrulaması yapın
auth.authenticate_user()
print("Google Drive ve BigQuery bağlantısı başarıyla kuruldu!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive ve BigQuery bağlantısı başarıyla kuruldu!


In [3]:
from google.cloud import bigquery

client = bigquery.Client(project='data426-group3')


query = """
    SELECT * FROM `data426-group3.dbt_Crnkcbs.mart_data426_group3_customer_behavior_analysis`
"""

print("Veri BigQuery'den çekiliyor, lütfen bekleyin...")
# Veriyi çekip Pandas DataFrame'e dönüştürüyoruz
df = client.query(query).to_dataframe()

# Drive'a kaydedilecek dosya adını temiz ve .csv uzantılı yapın
drive_kayit_yolu = '/content/drive/MyDrive/mart_data426_group3_customer_behavior_analysis.csv'

df.to_csv(drive_kayit_yolu, index=False)

print(f"Veri başarıyla Google Drive'a kaydedildi! Dosya Yolu: {drive_kayit_yolu}")

Veri BigQuery'den çekiliyor, lütfen bekleyin...
Veri başarıyla Google Drive'a kaydedildi! Dosya Yolu: /content/drive/MyDrive/mart_data426_group3_customer_behavior_analysis.csv


In [4]:
df.head()

,transaction_id,step,type,nameorig,amount,old_balance_orig,new_balance_orig,namedest,old_balance_dest,new_balance_dest,...,new_receiver_flag,many_to_one_receiver_flag,suspicious_cashout_flag,custom_month_id,monthly_transfer_volume,is_liquidation,transfer_drain_ratio,amount_to_monthly_volume_ratio,calculated_risk_score,risk_segment
0,TX_005681670,397,TRANSFER,C64945308,64066.95,15744.0,0.0,C759032940,309289.89,373356.85,...,1,0,0,1,64066.95,0,4.0693,1.0,35,Yüksek Risk - Manuel İnceleme
1,TX_005732012,399,TRANSFER,C2074014392,357786.37,0.0,0.0,C304147314,2759450.43,3117236.80,...,1,0,0,1,357786.37,0,0.0000,1.0,20,Orta Risk - Ek SMS Onayı
2,TX_004929997,350,TRANSFER,C1128254188,555584.89,0.0,0.0,C680017491,1887174.51,2442759.40,...,1,0,0,1,555584.89,0,0.0000,1.0,20,Orta Risk - Ek SMS Onayı
3,TX_005253566,371,TRANSFER,C188230044,200620.21,0.0,0.0,C1774621805,200762.08,401382.29,...,1,0,0,1,200620.21,0,0.0000,1.0,20,Orta Risk - Ek SMS Onayı
4,TX_005549974,382,TRANSFER,C118442842,2541563.77,0.0,0.0,C664718276,2798691.90,5340255.67,...,1,0,0,1,2541563.77,0,0.0000,1.0,30,Yüksek Risk - Manuel İnceleme


#HİPOTEZ ANALİZLERİ

 ## İşlem Türü ve Dolandırıcılık İlişkisi

İşlem türü (`type`) ile dolandırıcılık (`is_fraud`) arasında ilişki olup olmadığını **Ki-Kare Bağımsızlık Testi** ile test edeceğiz.

**Hipotezlerin Kurulması**
$H_0$ (Sıfır Hipotezi): İşlem türü ile bir işlemin fraud olması arasında hiçbir ilişki yoktur (Bu iki değişken birbirinden tamamen bağımsızdır).

$H_1$ (Alternatif Hipotez): İşlem türü ile bir işlemin fraud olması arasında anlamlı bir ilişki vardır (Bağımlıdırlar).

In [5]:
import pandas as pd
from scipy.stats import chi2_contingency

# 1. Adım: İki değişken arasında bir çapraz tablo (Contingency Table) oluşturuyoruz

contingency_table = pd.crosstab(df['type'], df['is_fraud'])

print("--- Çapraz Tablo (Gözlenen Değerler) ---")
print(contingency_table)
print("-" * 40)

# 2. Adım: Ki-Kare Testini çalıştırıyoruz
chi2, p, dof, expected = chi2_contingency(contingency_table)

print(f"Ki-Kare İstatistiği (Chi2): {chi2:.4f}")
print(f"P-Değeri (p-value): {p}")
print(f"Serbestlik Derecesi (dof): {dof}")

print("-" * 40)
# 3. Adım: Sonucu yorumlama
alpha = 0.05 # Standart güven sınırı (%95 güven)

if p < alpha:
    print("Sonuç: H0 hipotezi REDDEDİLDİ!")
    print("İşlem türü (type) ile dolandırıcılık (is_fraud) arasında İSTATİSTİKSEL OLARAK ANLAMLI BİR İLİŞKİ VARDIR.")
else:
    print("Sonuç: H0 hipotezi reddedilemedi.")
    print("İşlem türü ile dolandırıcılık arasında anlamlı bir ilişki bulunamamıştır (Bağımsızdırlar).")

--- Çapraz Tablo (Gözlenen Değerler) ---
is_fraud        0     1
type                   
CASH_IN   1399284     0
CASH_OUT  2233384  4116
DEBIT       41432     0
PAYMENT   2151495     0
TRANSFER   528812  4097
----------------------------------------
Ki-Kare İstatistiği (Chi2): 22082.5357
P-Değeri (p-value): 0.0
Serbestlik Derecesi (dof): 4
----------------------------------------
Sonuç: H0 hipotezi REDDEDİLDİ!
İşlem türü (type) ile dolandırıcılık (is_fraud) arasında İSTATİSTİKSEL OLARAK ANLAMLI BİR İLİŞKİ VARDIR.


**Çapraz Tablonun Analizi**

- `CASH_IN`, `DEBIT` ve `PAYMENT` işlemlerinde hiç fraud yok.
- 8.213 fraud vakasının tamamı iki türde toplanıyor: `CASH_OUT` (4.116) ve `TRANSFER` (4.097).
- Fraud oranı `TRANSFER`'de %0,77, `CASH_OUT`'ta %0,18'dir (genel oran %0,13).

**Ki-Kare sonucu:** χ² = 22.082,54, serbestlik derecesi = 4, p ≈ 0. Fraud'un işlem türünden bağımsız olduğu hipotezi reddedilir.

Not: Örneklem 6,3 milyon olduğu için neredeyse her fark anlamlı çıkar. Etki büyüklüğü (Cramér's V ≈ 0,06) küçüktür, çünkü fraud çok nadirdir. Bu yüzden p-değerinden çok yukarıdaki oranlar bilgilendiricidir.

##Hesap Bakiyesinin Sıfırlanması (Kritik Davranış Analizi)

In [6]:
import pandas as pd

# 1. Sadece fraud olan işlemleri filtreleyelim
fraud_df = df[df['is_fraud'] == 1].copy()

# 2. KRİTİK DÜZELTME: Başlangıç bakiyesi 0 olanları analiz dışı bırakıyoruz!
# Sadece içinde para olan (old_balance_orig > 0) hesapları alıyoruz.
real_balance_fraud = fraud_df[fraud_df['old_balance_orig'] > 0].copy()

# 3. Hesap tamamen sıfırlandı mı kontrolü (Artık tamamen güvenli)
real_balance_fraud['is_fully_drained'] = real_balance_fraud['amount'] == real_balance_fraud['old_balance_orig']

# 4. Yüzdelik oranları hesaplayalım
drain_counts = real_balance_fraud['is_fully_drained'].value_counts()
drain_percentages = real_balance_fraud['is_fully_drained'].value_counts(normalize=True) * 100

print(f"--- Gerçek Bakiyeli Hesaplarda ({len(real_balance_fraud)} vaka) Sıfırlama Davranışı ---")
for idx in drain_counts.index:
    durum = "Hesap Tamamen Sıfırlandı (All-in)" if idx else "Kısmi Vurgun / Bakiye Kalmış"
    print(f"{durum}: {drain_counts[idx]} vaka ({drain_percentages[idx]:.2f}%)")

print("-" * 50)

--- Gerçek Bakiyeli Hesaplarda (8172 vaka) Sıfırlama Davranışı ---
Hesap Tamamen Sıfırlandı (All-in): 8018 vaka (98.12%)
Kısmi Vurgun / Bakiye Kalmış: 154 vaka (1.88%)
--------------------------------------------------


Normal işlemlerin hesap sıfırlama oranı ile Fraud işlemlerin hesap sıfırlama oranı arasındaki fark istatistiksel olarak anlamlı mı? bunun için Z-Testi (İki Örneklem Oran Z-Testi - Two-Sample Proportion Z-Test) yapılacak.

**Z-Testi (İki Örneklem Oran Z-Testi - Two-Sample Proportion Z-Test)**

Hipotezlerin Kurulması

$H_0$ (Sıfır Hipotezi): Normal işlemlerin hesap boşaltma oranı ile fraud işlemlerin hesap boşaltma oranı arasında hiçbir fark yoktur($p_1 = p_2$).

$H_1$ (Alternatif Hipotez): Fraud işlemlerin hesap boşaltma oranı, normal işlemlerin oranından anlamlı derecede daha yüksektir ($p_1 > p_2$).

In [7]:
import numpy as np
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

# 1. Adım: Sadece başlangıç bakiyesi 0'dan büyük olan gerçek hesapları filtreleyelim
valid_accounts = df[df['old_balance_orig'] > 0].copy()

# 2. Adım: Grupları ayıralım (Fraud olanlar ve normal olanlar)
fraud_group = valid_accounts[valid_accounts['is_fraud'] == 1].copy()
normal_group = valid_accounts[valid_accounts['is_fraud'] == 0].copy()

# 3. Adım: Her iki grupta "Hesabı Sıfırlayan" (All-in) işlem sayılarını bulalım
fraud_group['is_drained'] = fraud_group['amount'] == fraud_group['old_balance_orig']
normal_group['is_drained'] = normal_group['amount'] == normal_group['old_balance_orig']

success_fraud = fraud_group['is_drained'].sum()    # Fraud olup hesabı sıfırlayanlar
success_normal = normal_group['is_drained'].sum()  # Normal olup hesabı sıfırlayanlar

n_fraud = len(fraud_group)    # Toplam gerçek bakiyeli fraud sayısı
n_normal = len(normal_group)  # Toplam gerçek bakiyeli normal işlem sayısı

# 4. Adım: Oranları ekrana yazdıralım
print(f"Fraud Grubu Sıfırlama Oranı : %{(success_fraud / n_fraud)*100:.2f} ({success_fraud}/{n_fraud})")
print(f"Normal Grup Sıfırlama Oranı : %{(success_normal / n_normal)*100:.2f} ({success_normal}/{n_normal})")
print("-" * 50)

# 5. Adım: İki Örneklem Oran Z-Testini Çalıştıralım (Alternative='larger' çünkü fraud'un büyük olmasını bekliyoruz)
counts = np.array([success_fraud, success_normal])
nobs = np.array([n_fraud, n_normal])

z_stat, p_value = proportions_ztest(counts, nobs, alternative='larger')

print(f"Z-İstatistiği (Z-score): {z_stat:.4f}")
print(f"P-Değeri (p-value): {p_value}")
print("-" * 50)

# 6. Adım: Sonucu Yorumlama
alpha = 0.05
if p_value < alpha:
    print("Sonuç: H0 hipotezi güçlü bir şekilde REDDEDİLDİ!")
    print("Saldırganların hesap kurutma/sıfırlama oranı, normal kullanıcıların oranından İSTATİSTİKSEL OLARAK ANLAMLI DERECEDE YÜKSEKTİR.")
else:
    print("Sonuç: H0 hipotezi reddedilemedi. İki grup arasında anlamlı bir oran farkı yoktur.")

Fraud Grubu Sıfırlama Oranı : %98.12 (8018/8172)
Normal Grup Sıfırlama Oranı : %0.00 (0/4251999)
--------------------------------------------------
Z-İstatistiği (Z-score): 2044.4406
P-Değeri (p-value): 0.0
--------------------------------------------------
Sonuç: H0 hipotezi güçlü bir şekilde REDDEDİLDİ!
Saldırganların hesap kurutma/sıfırlama oranı, normal kullanıcıların oranından İSTATİSTİKSEL OLARAK ANLAMLI DERECEDE YÜKSEKTİR.


**Çıktının Analizi**

| | Normal | Fraud |
|---|---|---|
| `is_liquidation = 0` | 6.354.407 | 195 |
| `is_liquidation = 1` | 0 | 8.018 |

- Başlangıç bakiyesi 0'dan büyük fraud işlemlerinin %98,12'si (8.018 / 8.172) hesabı tamamen boşaltıyor. Normal işlemlerde bu oran %0 (0 / 4.251.999).
- Z-testi (Z = 2044) farkı doğrular ama tabloya yeni bir bilgi eklemez.
- Kalan 154 fraud (%1,88) kısmi çekimdir. Analiz dışı bırakılan başlangıç bakiyesi 0 olan 41 fraud işlemi de vardır.
- `is_liquidation == 1` kuralı tek başına **precision 1,00** ve tüm veride **recall %97,6** (8.018 / 8.213) verir.
- Veri sentetik (PaySim) ve simülatörde fraud, hesabın boşaltılması senaryosuyla üretilmiştir. Bu yüzden bu kalıp gerçek bankacılık verisinde beklenmez.

#MACHINE LEARNING

## Risk Skoru Dağılımı

In [19]:
# Hangi risk puanında ne kadar fraud var yüzdesel dağılımı
print(df.groupby('is_fraud')['calculated_risk_score'].describe())

              count       mean        std   min   25%   50%   75%   max
is_fraud                                                               
0         6354407.0  22.220873   8.282824  15.0  15.0  20.0  30.0  75.0
1            8213.0  45.664191  10.465027  20.0  40.0  40.0  45.0  85.0


**Risk skoru dağılımı**

- **Ortalama:** Fraud işlemlerin ortalama skoru (45,7), normal işlemlerin (22,2) yaklaşık 2 katıdır.
- **Minimum:** En düşük skorlu fraud işlemi 20 puan almıştır (normal işlemlerde minimum 15).
- **Medyan:** Normal işlemlerin yaklaşık %43'ü 15 puanda kalmış, medyanları 20'dir. Fraud işlemlerin medyanı 40'tır.
- **Maksimum:** Normal işlemlerde en yüksek skor 75, fraud işlemlerde 85'tir. Normal işlemler arasında da yüksek skor alanlar vardır.

##Ana eşik değeri belirleme

    **Eşik Değerleri**
    
    calculated_risk_score >= 50 THEN 'Çok Yüksek Risk - Blokaj / Alarm'
    calculated_risk_score >= 25 THEN 'Yüksek Risk - Manuel İnceleme'
    calculated_risk_score >= 15 THEN 'Orta Risk - Ek SMS Onayı'
    'Düşük Risk - Güvenli İşlem'

**Segmentlerin gerçek performansı**

| Segment | İşlem | Fraud | Fraud oranı | Tüm fraud'un payı |
|---|---|---|---|---|
| Orta Risk (15–24) | 4.265.819 | 39 | %0,001 | %0,5 |
| Yüksek Risk (25–49) | 2.077.390 | 6.335 | %0,305 | %77,1 |
| Çok Yüksek Risk (≥50) | 19.411 | 1.839 | %9,47 | %22,4 |

- **≥50 kesin blokaj için uygun değildir.** Bu segmentteki 19.411 işlemin 17.572'si normal işlemdir.
- Yüksek Risk segmenti işlemlerin yaklaşık üçte biridir, hepsini manuel incelemek pratik değildir.
- "Düşük Risk" segmenti boştur: minimum skor 15'tir. `is_new_receiver_for_sender` ve `new_receiver_flag` tüm satırlarda sabit olduğundan her işlem büyük olasılıkla 15 puanla başlıyor.
- Skor ile fraud oranı doğrusal artmaz (skor 40'ta %64,7; skor 55'te %1,9; skor 75'te %94,8).
- Sonuç: bu skor tek başına karar mekanizması olarak zayıftır. Sonraki bölümde "önce kural, sonra model" yaklaşımı kuruldu.

#BİNARY CLASSIFICATION İLE MODEL EĞİTME

## LightGBM Modelini Eğitmek

## Deney 1: İlk Model (12 sütun, risk skoru dahil)

### Parametreleri Belirlemek


**Belirlenen riskli müşteri davranışları ile fraud hesaplarını yakalamak**

*   Hedef (Target) Y= İs_fraud
*   Özellikler(Features) X = 	calculated_risk_score(X1),,type_code(X2),
amount_to_monthly_volume_ratio(x3),is_liquidation(X4),many_to_one_receiver_flag(X5),is_new_receiver_for_sender(X6),new_receiver_flag(X7),high_amount_flag(X8),high_velocity_flag(X9),is_night(X10),transfer_drain_ratio(X11),suspicious_cashout_flag(X12)

*   Denetimli (supervised) Öğrenmedir.

*   Veriler yapılandırılmıştır.

*   Binary Classification (İkili Sınıflandırma).

*   LightGBM ağaç tabanlı (Tree-based) modelli






### Deney 1: İlk Model (12 sütun, risk skoru dahil)

In [10]:
## Deney 1: İlk Model (12 sütun, risk skoru dahil)


import os
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# 1. df değişkeninin tanımlı olup olmadığını kontrol edelim, yoksa Drive'dan yükleyelim
if 'df' not in locals() and 'df' not in globals():
    drive_kayit_yolu = '/content/drive/MyDrive/mart_data426_group3_customer_behavior_analysis.csv'
    if os.path.exists(drive_kayit_yolu):
        print("df bulunamadı, veriler Google Drive'dan yükleniyor...")
        df = pd.read_csv(drive_kayit_yolu)
    else:
        raise NameError("df tanımlı değil ve Google Drive'da yedek dosya bulunamadı! Lütfen önce veriyi BigQuery'den çeken hücreyi çalıştırın.")

# Özellikleri (X) ve Hedefi (Y) tanımlama
features = [
    'type_code', 'amount_to_monthly_volume_ratio','transfer_drain_ratio','is_liquidation',
    'many_to_one_receiver_flag', 'suspicious_cashout_flag','is_new_receiver_for_sender', 'new_receiver_flag',
    'high_amount_flag', 'high_velocity_flag', 'is_night', 'calculated_risk_score'
]

X = df[features].astype(float)
y = df['is_fraud']

# 2. Veriyi %80 Eğitim, %20 Test olarak bölüyoruz (Stratify ile fraud oranını koruyoruz)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. LightGBM Modelini Tanımlıyoruz (Dengesiz veri seti ayarı aktif)
model = lgb.LGBMClassifier(
    objective='binary',
    is_unbalance=True,  # 6.3 milyon verideki fraud azlığını dengelemek için kritik parametre!
    n_estimators=100,
    learning_rate=0.05,
    random_state=42
)

# 4. Modeli Eğitiyoruz
print("Model eğitiliyor, lütfen bekleyin...")
model.fit(X_train, y_train)
print("Eğitim tamamlandı!")

Model eğitiliyor, lütfen bekleyin...
[LightGBM] [Info] Number of positive: 6570, number of negative: 5083526
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.350530 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 305
[LightGBM] [Info] Number of data points in the train set: 5090096, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.001291 -> initscore=-6.651247
[LightGBM] [Info] Start training from score -6.651247
Eğitim tamamlandı!


### Modelin Performans Ölçümü

In [5]:
## Deney 1: Performans Ölçümü

import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# 1. Test verisi üzerinde tahminler
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# 2. Sonuç Raporları (Classification Report & Confusion Matrix)
print("=== MODEL: CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))

print("=== MODEL: CONFUSION MATRIX ===")
print(confusion_matrix(y_test, y_pred))

print(f"\nYeni ROC-AUC Skoru: {roc_auc_score(y_test, y_pred_proba):.4f}")

# 3. Puanlama Sütununun (Calculated Risk Score) Ölçüm
importance = pd.DataFrame({'Özellik (Feature)': features, 'Önem Derecesi (Importance)': model.feature_importances_})
importance = importance.sort_values('Önem Derecesi (Importance)', ascending=False)
print("\n=== ÖZELLİK ÖNEM DERECELERİ ===")
print(importance)

=== MODEL: CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

           0       1.00      0.94      0.97   1270881
           1       0.02      0.99      0.04      1643

    accuracy                           0.94   1272524
   macro avg       0.51      0.96      0.50   1272524
weighted avg       1.00      0.94      0.97   1272524

=== MODEL: CONFUSION MATRIX ===
[[1188482   82399]
 [      9    1634]]

Yeni ROC-AUC Skoru: 0.9657

=== ÖZELLİK ÖNEM DERECELERİ ===
                 Özellik (Feature)  Önem Derecesi (Importance)
2             transfer_drain_ratio                        2011
0                        type_code                         430
11           calculated_risk_score                         223
1   amount_to_monthly_volume_ratio                         166
10                        is_night                          73
4        many_to_one_receiver_flag                          37
5          suspicious_cashout_flag                          34


In [11]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score

# Test etmek istediğimiz eşik değer aralıkları
thresholds = np.arange(0.1, 1.0, 0.1)
results = []
for t in thresholds:
    preds = (y_pred_proba >= t).astype(int)
    p = precision_score(y_test, preds)
    r = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    results.append({"Threshold": round(t, 2), "Precision": p, "Recall": r, "F1-Score": f1})
# Sonuçları net bir tablo olarak görelim
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

 Threshold  Precision   Recall  F1-Score
       0.1   0.018934 0.994522  0.037160
       0.2   0.019316 0.994522  0.037896
       0.3   0.019316 0.994522  0.037896
       0.4   0.019445 0.994522  0.038144
       0.5   0.019445 0.994522  0.038144
       0.6   0.019445 0.994522  0.038144
       0.7   0.019445 0.994522  0.038144
       0.8   0.019445 0.994522  0.038144
       0.9   0.019445 0.994522  0.038144


In [8]:
import numpy as np
from sklearn.metrics import precision_score, recall_score

# Farklı eşik değerlerini test ediyoruz
for threshold in [0.15, 0.25, 0.50, 0.70]:
    # Eğer modelin fraud olasılığı eşik değerinden büyükse 1 (fraud) kabul et
    temp_preds = np.where(y_pred_proba >= threshold, 1, 0)

    print(f"--- Eşik Değeri (Threshold): {threshold} ---")
    print(f"Yakalama Oranı (Recall): {recall_score(y_test, temp_preds):.2f}")
    print(f"Doğruluk Oranı (Precision): {precision_score(y_test, temp_preds):.2f}\n")

--- Eşik Değeri (Threshold): 0.15 ---
Yakalama Oranı (Recall): 0.99
Doğruluk Oranı (Precision): 0.02

--- Eşik Değeri (Threshold): 0.25 ---
Yakalama Oranı (Recall): 0.99
Doğruluk Oranı (Precision): 0.02

--- Eşik Değeri (Threshold): 0.5 ---
Yakalama Oranı (Recall): 0.99
Doğruluk Oranı (Precision): 0.02

--- Eşik Değeri (Threshold): 0.7 ---
Yakalama Oranı (Recall): 0.99
Doğruluk Oranı (Precision): 0.02



## Deney 2: Kural Baseline'ı + Düzeltilmiş Model

Deney 1'de model çok fazla yanlış alarm üretti (precision 0,019). Deney 2'de üç şey değiştirildi:

1. **Özellikler temizlendi:** Tüm satırlarda aynı değeri taşıyan üç bayrak ve `calculated_risk_score` çıkarıldı.
2. **Basit bir baseline eklendi:** Modelin gerçekten bir katkısı olup olmadığını görmek için `is_liquidation == 1` kuralıyla karşılaştırıldı.
3. **Eğitim kararlı hale getirildi:** Doğrulama seti ile erken durdurma, `scale_pos_weight=10`, `min_child_samples=50` ve `reg_lambda=1` kullanıldı.

### Parametreleri Belirlemek

**Belirlenen riskli müşteri davranışları ile fraud hesaplarını yakalamak**
- **Hedef (y):** `is_fraud`
- **Özellikler (8):** `type_code`, `amount_to_monthly_volume_ratio`, `transfer_drain_ratio`, `is_liquidation`, `many_to_one_receiver_flag`, `suspicious_cashout_flag`, `high_amount_flag`, `is_night`
- **Çıkarılanlar:**
  - `is_new_receiver_for_sender`, `new_receiver_flag`, `high_velocity_flag`: tüm satırlarda aynı değer, bilgi taşımıyor
  - `calculated_risk_score`: aynı bayrakların puan toplamı, yeni bilgi eklemiyor
- **Bölme:** Deney 1 ile aynı (%80 eğitim / %20 test, `random_state=42`, `stratify`). Eğitim setinin %20'si ayrıca doğrulama için ayrıldı.
- **Metrik:** PR-AUC, precision, recall ve yanlış alarm sayısı

In [12]:
const_cols = [c for c in features if X_train[c].nunique() == 1]
exclude = const_cols + ['calculated_risk_score']
feats_clean = [f for f in features if f not in exclude]

print("Sabit sütunlar:", const_cols)
print("Modelde kullanılacak:", feats_clean)
print("Özellik sayısı:", len(feats_clean))

Sabit sütunlar: ['is_new_receiver_for_sender', 'new_receiver_flag', 'high_velocity_flag']
Modelde kullanılacak: ['type_code', 'amount_to_monthly_volume_ratio', 'transfer_drain_ratio', 'is_liquidation', 'many_to_one_receiver_flag', 'suspicious_cashout_flag', 'high_amount_flag', 'is_night']
Özellik sayısı: 8


### Kural Baseline'ı

Fraud'u tek bir kuralla yakalayabilir miyiz? `is_liquidation == 1` (hesabın tamamının boşaltılması) kuralı test setinde denendi.

In [13]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import precision_score, recall_score

y_te = y_test.astype(int).values
rule = X_test['is_liquidation'].values.astype(int)

print(confusion_matrix(y_te, rule))
print("precision:", round(precision_score(y_te, rule), 4))
print("recall   :", round(recall_score(y_te, rule), 4))

[[1270881       0]
 [     38    1605]]
precision: 1.0
recall   : 0.9769


### Kuralın Kaçırdığı Fraud'lar

Kural fraud'un yaklaşık %2,6'sını kaçırıyor. Bu işlemlerin ortak özelliği nedir?

In [14]:
rest = df[df['is_liquidation'] == 0]
miss = rest[rest['is_fraud'] == 1]
sifir = (miss['old_balance_orig'] == 0).sum()
tavan = (miss['amount'] >= 10_000_000).sum()

print("Kuralın kaçırdığı fraud (tüm veri):", len(miss))
print(miss['type'].value_counts())
print("Başlangıç bakiyesi 0 olanlar:", sifir)
print("Tutarı 10 milyon ve üzeri olanlar:", tavan)
print("Tüm veride en yüksek tutar:", df['amount'].max())

Kuralın kaçırdığı fraud (tüm veri): 195
type
TRANSFER    154
CASH_OUT     41
Name: count, dtype: int64
Başlangıç bakiyesi 0 olanlar: 41
Tutarı 10 milyon ve üzeri olanlar: 145
Tüm veride en yüksek tutar: 92445516.64


### Model Eğitimi

Model, kuralın yetersiz kaldığı yerde bir katkı sağlıyor mu? LightGBM, doğrulama setinde logloss iyileşmeyi bırakınca duracak şekilde eğitildi.

In [15]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train[feats_clean], y_train,
    test_size=0.2, random_state=42, stratify=y_train)

m2 = lgb.LGBMClassifier(
    objective='binary',
    n_estimators=2000,
    learning_rate=0.05,
    scale_pos_weight=10,
    min_child_samples=50,
    reg_lambda=1.0,
    random_state=42,
    verbose=-1)

m2.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    eval_metric='binary_logloss',
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])

print("En iyi iterasyon:", m2.best_iteration_)

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[170]	valid_0's binary_logloss: 0.000166444
En iyi iterasyon: 170


### Modelin Performans Ölçümü

### Karşılaştırma

Üç yaklaşım aynı test setinde karşılaştırıldı: yalnızca kural, kural + model (eşik 0,5) ve kural + model (eşik 0,9). Eşik 0,5 sonuçlara bakılmadan seçilen varsayılan değerdir, 0,9 yalnızca duyarlılık kontrolü içindir.

In [16]:
from sklearn.metrics import average_precision_score
import pandas as pd

p2 = m2.predict_proba(X_test[feats_clean])[:, 1]
liq = rule

print("Genel PR-AUC:", round(average_precision_score(y_te, p2), 4))

mask = liq == 0
print("Kural dışı fraud sayısı:", y_te[mask].sum())
print("Kural dışı PR-AUC:",
      round(average_precision_score(y_te[mask], p2[mask]), 4))
print("Rastgele PR-AUC ≈", round(y_te[mask].mean(), 6))

for t in [0.5, 0.9]:
    hyb = ((liq == 1) | (p2 >= t)).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te, hyb).ravel()
    print(f"eşik={t}: precision={tp/(tp+fp):.3f}",
          f"recall={tp/(tp+fn):.3f}",
          f"yanlış alarm={fp} kaçan={fn}")

imp = pd.Series(m2.booster_.feature_importance('gain'), index=feats_clean)
print((imp / imp.sum()).sort_values(ascending=False).round(3))

Genel PR-AUC: 0.9957
Kural dışı fraud sayısı: 38
Kural dışı PR-AUC: 0.6982
Rastgele PR-AUC ≈ 3e-05
eşik=0.5: precision=0.996 recall=0.996 yanlış alarm=7 kaçan=7
eşik=0.9: precision=0.998 recall=0.991 yanlış alarm=4 kaçan=14
is_liquidation                    0.992
transfer_drain_ratio              0.007
is_night                          0.001
high_amount_flag                  0.000
type_code                         0.000
amount_to_monthly_volume_ratio    0.000
many_to_one_receiver_flag         0.000
suspicious_cashout_flag           0.000
dtype: float64


### Deney 2 Sonuçları

| Yaklaşım | Precision | Recall | Yanlış alarm | Kaçan fraud |
|---|---|---|---|---|
| Yalnızca kural (`is_liquidation == 1`) | 1,000 | 0,977 | 0 | 38 |
| Kural + model (eşik 0,5) | 0,996 | 0,996 | 7 | 7 |
| Kural + model (eşik 0,9) | 0,998 | 0,991 | 4 | 14 |

- Genel PR-AUC 0,996'dır (yalnızca kuralın PR-AUC'si 0,977).
- Deney 1'e göre yanlış alarm sayısı 82.399'dan 7'ye düştü.
- Kuralın kaçırdığı 38 test fraud'undan model, eşik 0,5'te 31'ini yakaladı.
- Kural dışı işlemlerde modelin PR-AUC'si 0,70'tir (rastgele tahmin ≈ 0,00003). Model, kuralın kaçırdığı fraud'ların bir kısmını gerçekten ayırt edebiliyor.
- Model büyük ölçüde `is_liquidation` ve `transfer_drain_ratio` özelliklerine dayanıyor. Düşük önemli görünen özellikler zor vakalarda rol oynuyor olabilir.
- markdown
- Kuralın kaçırdığı 195 fraud'un 154'ü kısmi çekimdir, 41'inin başlangıç bakiyesi 0'dır. Kısmi çekimlerin 145'i (%94) tam 10.000.000 tutarındadır ve 148'inde hesap bakiyesi 10 milyondan yüksektir. Bu, hesabın tamamının tek seferde boşaltılamadığı, tutarın 10 milyonla sınırlandığı bir durumla uyumludur
- **Tutar gözlemi:** Fraud işlemlerinde en yüksek tutar tam 10.000.000'dur (287 fraud işlemi tam bu tutardadır). Normal işlemlerde tutar 92 milyona kadar çıkar (2.920 işlem tam 10M, 2.443 işlem 10M üzeri). Bu, fraud tutarlarının simülasyonda 10 milyonla sınırlandırıldığını düşündürür. `amount == 10M` tek başına kural olarak kullanılamaz (precision ≈ %9), ve tutar bilgisi modele özellik olarak verilmemiştir

- Fraud tutarlarının tam 10 milyonda sınırlanması (normal işlemlerde sınır yok), verinin sentetik üretim kuralından kaynaklanıyor olabilir. Bu örüntü gerçek bankacılık verisinde beklenmez.

**Sınırlamalar**
- Kural dışı test grubunda yalnızca 38 fraud var. Eşik 0,5 ile 0,9 arasındaki fark (7 ve 4 yanlış alarm) bu örneklemle gürültü düzeyindedir.
- Test seti satır sırasına bağlı olarak değişebiliyor. Aynı veri dosyası kullanıldığında sonuçlar tekrarlanabilir.
- Veri sentetik. Fraud'un "hesabı tamamen boşalt" kalıbıyla üretilmiş olması, kuralın bu kadar güçlü çıkmasını açıklayabilir. Sonuçlar gerçek bankacılık verisine doğrudan taşınamaz.
- Eşik (0,5) sonuçlara bakılmadan seçilen varsayılan değerdir, 0,9 yalnızca karşılaştırma içindir.

In [17]:
partial = miss[miss['old_balance_orig'] > 0]
print("Kısmi çekim sayısı:", len(partial))
print("Tutarı tam 10M olanlar:", (partial['amount'] == 10_000_000).sum())
print("Bakiyesi 10M'den büyük olanlar:", (partial['old_balance_orig'] > 10_000_000).sum())

Kısmi çekim sayısı: 154
Tutarı tam 10M olanlar: 145
Bakiyesi 10M'den büyük olanlar: 148


In [18]:
fr = df[df['is_fraud'] == 1]
nm = df[df['is_fraud'] == 0]

print("Fraud'da en yüksek tutar:", fr['amount'].max())
print("Fraud'da tam 10M olan:", (fr['amount'] == 10_000_000).sum())
print("Normalde tam 10M olan:", (nm['amount'] == 10_000_000).sum())
print("Normalde 10M ve üzeri:", (nm['amount'] >= 10_000_000).sum())

Fraud'da en yüksek tutar: 10000000.0
Fraud'da tam 10M olan: 287
Normalde tam 10M olan: 2920
Normalde 10M ve üzeri: 5363
